# Lyden M11 — CLM memory continue-train + MoE balancer fix (Kaggle 2× T4 DDP)

Config: `configs/m11-100m-continue-ddp.yaml` — resumes the M10 ckpt
(ckpt_0007600.pt, `runs/100m/ckpts` in this notebook's own previous OUTPUT),
fresher schedule over **2,000 more steps** (~65.5M tokens: 70% TinyStories +
~27% synthetic fact-fix CLM sessions via `scripts/m11_prepare_sessions.py`,
ratio 2). Model geometry unchanged (window 1024). M11 wraps three fixes:

1. **Balancer actually running now** (`Trainer.step` calls `update_bias`
   per step — M10's imbalance root cause was the missing call site).
2. Router z-norm + gamma ramp `(0.004 → 0.001 @400)` + fresh bias=0 on load.
3. Success bar: held-out fact-fix recall ≥ 0.80, story ppl ≤ 15,
   load_top ≤ 0.5 / entropy ≥ 1.95 in all 4 routed layers.

Run policy: **Save Version → Save & Run All (Commit)**, GPU **T4 ×2**, Internet ON,
**private dataset `chutpaglubandahure/lyden-100m-backup` attached (ckpt_0007600.pt + tokenizer).** New notebook — the old one stays untouched.

Continuation/sessions: cell 3 restores `runs/100m-m11/ckpts` from the previous
output and auto-resumes from there; `--resume` (cell 5) is the cross-phase
entry that only fires when no own-m11 ckpts exist.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import sys
print(sys.version)

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /kaggle/working/Lyden/.git ]; then
  git clone https://github.com/mrrchandra/Lyden.git /kaggle/working/Lyden
else
  cd /kaggle/working/Lyden && git pull --ff-only || echo '(offline keep)'
fi
ENTRY=$(find /kaggle/input -name 'ckpt_0007600.pt' 2>/dev/null | head -1 || true)
TOKJ=$(find /kaggle/input -name 'tokenizer.json' 2>/dev/null | head -1 || true)
if [ -z "$ENTRY" ] || [ -z "$TOKJ" ]; then
  echo '[FATAL] backup dataset missing: ENTRY='"$ENTRY"' TOKJ='"$TOKJ"
  echo '        attach dataset chutpaglubandahure/lyden-100m-backup and re-run.'
  exit 1
fi
mkdir -p /kaggle/working/ckpts-m10 /kaggle/working/lyden-data/tokenizer
cp "$ENTRY" /kaggle/working/ckpts-m10/ckpt_0007600.pt
cp "$TOKJ".json /kaggle/working/lyden-data/tokenizer/tokenizer.json
TOKM=$(find /kaggle/input -name 'meta.json' 2>/dev/null | head -1 || true)
[ -n "$TOKM" ] && cp "$TOKM" /kaggle/working/lyden-data/tokenizer/meta.json || true
echo "[entry] restored ckpt from"
echo "        $ENTRY"
echo "[entry] tokens from $TOKJ"
ls -la /kaggle/working/ckpts-m10/ /kaggle/working/lyden-data/tokenizer/
cd /kaggle/working/Lyden
python - <<'PYEOF'
import sys
sys.path.insert(0, 'src')
from pathlib import Path
from lyden.config import load_config
if not Path('/kaggle/working/lyden-data/tokenizer/tokenizer.json').exists():
    sys.exit('[FATAL] tokenizer missing')
c = load_config('configs/m11-100m-continue-ddp.yaml')
t = c.training
print('config ok:', t.seq_len * t.batch_size * t.grad_accum * 2, 'tok/step;',
      t.total_steps, 'steps;', t.amp, '| reset_sched:', t.reset_scheduler,
      '| reset_bias:', t.reset_moe_bias, '| znorm:', c.model.moe.router_znorm)
PYEOF

In [ ]:
# Install mamba-ssm + causal-conv1d, fastest source first:
# 1. wheels cached in this notebook's own previous output (attached as Input)
# 2. prebuilt release wheels matched to this runtime's torch/CUDA/cpXX
# 3. source build with --no-build-isolation (pip's isolated build env has no
#    torch, hence the classic crash) — built wheels land in
#    /kaggle/working/wheels so future sessions never compile again.
import glob, os, subprocess, sys, urllib.request, torch

TV = '.'.join(torch.__version__.split('+')[0].split('.')[:2])   # e.g. '2.8'
if torch.version.cuda is None:
    raise RuntimeError(
        'no GPU in this session (torch.version.cuda is None).'
        ' Attach GPU T4 x2 on the notebook and re-run (Save & Run All).')
CU = torch.version.cuda.split('.')[0]                          # '12' or '13'
CP = f'cp{sys.version_info[0]}{sys.version_info[1]}'
print('runtime:', torch.__version__, 'cuda', CU, CP)

JOBS = [
    ('causal_conv1d', 'causal-conv1d==1.7.0',
     f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/'
     f'causal_conv1d-1.7.0+cu{CU}torch{TV}cxx11abiTRUE-{CP}-{CP}-linux_x86_64.whl'),
    ('mamba_ssm', 'mamba-ssm==2.3.2.post1',
     f'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/'
     f'mamba_ssm-2.3.2.post1%2Bcu{CU}torch{TV}cxx11abiTRUE-{CP}-{CP}-linux_x86_64.whl'),
]
for name, pkg, url in JOBS:
    cached = sorted(glob.glob(f'/kaggle/input/*/wheels/{name}-*.whl'))
    if cached:
        print(f'>> pip install (own-output cache) {cached[0]}')
        subprocess.check_call(['pip', 'install', '-q', cached[0]])
        continue
    ok = True
    try:
        urllib.request.urlopen(urllib.request.Request(url, method='HEAD'), timeout=20)
    except Exception as e:
        ok = False
        print(f'[wheel miss] {pkg}: {e}')
    if ok:
        subprocess.check_call(['pip', 'install', '-q', url])
    else:
        os.makedirs('/kaggle/working/wheels', exist_ok=True)
        print(f'>> pip wheel --no-build-isolation {pkg} (CUDA compile, up to ~45 min)')
        subprocess.check_call(['pip', 'wheel', '--no-build-isolation', '-q',
                               '-w', '/kaggle/working/wheels', pkg])
        wheel = sorted(glob.glob(f'/kaggle/working/wheels/{name}-*.whl'))[0]
        subprocess.check_call(['pip', 'install', '-q', wheel])

from mamba_ssm.modules.mamba2 import Mamba2
blk = Mamba2(d_model=512, d_state=128, d_conv=4, expand=2, layer_idx=0, use_mem_eff_path=False).cuda()
with torch.autocast('cuda', dtype=torch.float16):   # match training semantics
    y = blk(torch.randn(2, 256, 512, device='cuda'))
print('mamba-ssm OK:', y.shape, y.dtype, torch.__version__)


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Lyden
python scripts/m11_prepare_sessions.py \
  --tokenizer-dir /kaggle/working/lyden-data/tokenizer \
  --data-dir /kaggle/working/lyden-data-m11 \
  --max-train-tokens 65000000 \
  --max-val-tokens 700000 \
  --ratio 2
du -sh /kaggle/working/lyden-data-m11

In [ ]:
# Training: continue-train from the M10 ckpt (cross-phase). Auto-resume
# inside scripts/train.py picks up runs/100m-m11/ckpts on later rounds;
# --resume only fires when that dir is empty (entry condition).
import os, subprocess, sys, time

os.chdir('/kaggle/working/Lyden')
os.environ['PYTHONUNBUFFERED'] = '1'
CMD = [sys.executable, '-m', 'torch.distributed.run', '--standalone',
       '--nproc-per-node=2', 'scripts/train.py',
       '--config', 'configs/m11-100m-continue-ddp.yaml',
       '--resume', '/kaggle/working/ckpts-m10/ckpt_0007600.pt',
       '--val-every', '400']
for rnd in range(1, 9):
    print(f'[train round {rnd}] start', flush=True)
    proc = subprocess.Popen(CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    proc.wait()
    if proc.returncode == 0:
        print(f'[train round {rnd}] completed', flush=True)
        break
    print(f'[train round {rnd}] exited {proc.returncode}; retrying from last ckpt', flush=True)
    time.sleep(5)
else:
    print('[train] all rounds failed', flush=True)

In [ ]:
%%bash
set -uo pipefail
cd /kaggle/working/Lyden
CKPT=$(ls -1 runs/100m-m11/ckpts/ckpt_*.pt 2>/dev/null | sort | tail -1)
if [ -z "$CKPT" ]; then echo '[eval] no checkpoints yet; skipping'; exit 0; fi
echo "[eval] $CKPT"
python scripts/m11_eval.py \
    --config configs/m11-100m-continue-ddp.yaml \
    --ckpt "$CKPT" \
    --tokenizer-dir /kaggle/working/lyden-data/tokenizer \
    --eval-seeds 40 \
    --ppl-batches 20
echo '=== eval report ==='
cat runs/100m-m11/eval_report.json | head -80